# L7a: Utility-Based Allocation and the Adaptive Rebalancing Engine
In this lecture, we stop treating a portfolio as a vector of weights chosen once and treat it as a process. L5b and L6b chose minimum-variance and tangent portfolios on a decision date and held them; today we look at the other days. We introduce utility-based allocation, in which the investor's preference for each asset is written down explicitly and recomputed every day from the single index model (SIM) and the state of the market, and which answers two coupled questions at once: which assets belong in the basket today, and how much of each. We then wrap the allocator in a rebalancing engine: a daily loop with a self-financing account, next-bar execution, three trigger rules, and a scorecard.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Explain why a static allocation fails:__ Derive how buy-and-hold weights drift with prices and measure the drift, and state the three ways the minimum-variance inputs let the optimizer down: sensitivity to estimated means, concentration, and assumption fragility under a change of market state.
> * __Solve utility-based allocation problems:__ Pose the budget-constrained maximum-utility problem, solve the Cobb–Douglas problem in closed form, state the CES solution and its three limits, and compute the preference weights from the SIM parameters and two market inputs, reading their signs as the basket and their magnitudes as the weights.
> * __Build and score a rebalancing engine:__ Write the self-financing account recursion with transaction costs and next-bar execution, state the reallocation schedule, turnover cap, and drawdown limit, and score a realized path by portfolio NPV, drawdown, realized growth and Sharpe ratio, turnover, and cost.

Let's get started!
___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Building a maximum-utility portfolio allocator](CHEME-5660-L7a-Example-Utility-Allocator-Fall-2026.ipynb). Compute the market-state signal and the recent market growth from SPY, turn the SIM parameters of thirteen firms into preference weights, read which firms belong in the basket on the first trading day of 2025 (none) and on the first day the basket is nonempty, solve the Cobb–Douglas allocation in closed form and check it against the course package, sweep the CES elasticity, and compare with the L6b minimum-variance and tangent portfolios.

The second example wires the allocator into the engine:

> [▶ Running the adaptive rebalancing engine through 2025](CHEME-5660-L7a-Example-Adaptive-Rebalancing-Scorecard-Fall-2026.ipynb). Build the market inputs and the preference weights for every 2025 day with a one-day lag (asserted in code), run the engine on monthly and daily schedules with the turnover cap and drawdown limit on and off, and a CES variant with the elasticity rule, run the buy-and-hold GMV, tangent, equal-weight, and SPY portfolios through the same accounting, and read the realized-path scorecard.

The third example measures the drift that motivates all of this:

> [▶ Drift of maximum Sharpe ratio portfolios](CHEME-5660-L7a-Example-Portfolio-Drift-Fall-2026.ipynb). Compute the tangent portfolio from SIM inputs, hold its share counts fixed through 2025, and track the weights, the half-$L^{1}$ distance from the initial allocation, and the portfolio's $\alpha$ and $\beta$ as they drift.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

## Concept Review: Allocation as a Time-Stamped Decision
In L5b and L6b we chose weights $\mathbf{w}$ over $|\mathcal{P}|$ risky assets by solving a minimum-variance problem whose inputs were the expected growth-rate vector and the growth-rate covariance matrix. With the SIM of L6a those inputs are $\hat{\mathbf{g}}_{\text{SIM}} = \hat{\boldsymbol{\alpha}} + \hat{\boldsymbol{\beta}}\,g^{\prime}_{M}$ and $\hat{\mathbf{\Sigma}}_{g,\text{SIM}} = s^{2}_{g,M}\,\hat{\boldsymbol{\beta}}\hat{\boldsymbol{\beta}}^{\top} + \hat{\mathbf{D}}_{g}$, where $\hat{\alpha}_{i}$ and $\hat{\beta}_{i}$ are the estimated intercept and beta of asset $i$, $g^{\prime}_{M}$ and $s^{2}_{g,M}$ are the training-period sample mean and variance of the market growth rate, and $\hat{\mathbf{D}}_{g}$ holds the residual variances $s^{2}_{g,\varepsilon,i}$.

> __Recall (L5b, L6b):__ For a target growth rate $g_{\star}$, the long-only weights $\mathbf{w}(g_{\star})$ minimize $\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_{g}\mathbf{w}$ subject to $\mathbf{w}^{\top}\hat{\mathbf{g}} = g_{\star}$, $\mathbf{1}^{\top}\mathbf{w} = 1$, $w_{i}\geq 0$; the lowest point of the frontier is the global minimum-variance portfolio $\mathbf{w}_{\text{GMV}}$, and with a risk-free asset earning $g_{f}$ the fully invested point of the ray of risky and risk-free solutions is the tangent portfolio $\mathbf{w}_{\mathcal{T}}$, the risky portfolio with the largest Sharpe ratio (quoted annualized, as in L6b).

Whatever the choice, it is implemented on a decision date $t = 0$ by buying share counts $n_{i} = w_{i}\,W_{\mathcal{P}}(0)/S_{i}(0)$, where $W_{\mathcal{P}}(0)$ is the initial wealth and $S_{i}(0)$ the price of asset $i$ (L5a), and if nothing is traded afterward the wealth on a later day is the share-weighted price aggregate:
$$
W_{\mathcal{P}}(t) = \sum_{i\in\mathcal{P}}n_{i}\,S_{i}(t)
$$
Two things to say out loud: the weights are optimal on the decision date, for the inputs estimated on that date, and the only decision the investor ever made was the one at $t = 0$. This lecture is about the other days.
___

## Why Static Allocation Fails
A buy-and-hold portfolio fails to stay what it was for two separate reasons: its weights move mechanically with prices, and the inputs it was optimized against were never exactly right.

### Mechanical drift
With fixed share counts, no cash, and no dividends or other distributions, the weight of asset $i$ on day $t$ is its share of the portfolio's value, $w_{i}(t) = n_{i}S_{i}(t)/W_{\mathcal{P}}(t)$. Substituting $n_{i} = w_{i}(0)\,W_{\mathcal{P}}(0)/S_{i}(0)$ into numerator and denominator, the initial wealth cancels and the drifted weight depends only on the initial weights and the price relatives:
$$
w_{i}(t) = \frac{w_{i}(0)\,\overbrace{\left(S_{i}(t)/S_{i}(0)\right)}^{\text{price relative}}}{\sum_{j\in\mathcal{P}}w_{j}(0)\left(S_{j}(t)/S_{j}(0)\right)}
$$
The weights stay put only if every asset held with positive weight has the same price relative, which does not happen in practice: assets that outperform grow their weight and drag the portfolio's $\beta$ and concentration with them. We measure how far the portfolio has moved by the half-$L^{1}$ distance from the initial allocation:
$$
\underbrace{\tfrac{1}{2}\sum_{i\in\mathcal{P}}\left|w_{i}(t) - w_{i}(0)\right|}_{\text{drift on day }t}
$$
which, for a fully invested portfolio, is the fraction of wealth a rebalance back to the initial weights would have to trade on day $t$ before costs. The drift example finds that the SIM tangent portfolio held through 2025 ended the year with a drift of about 4.6%, with NVDA's weight moving between 0.49 and 0.58 and the portfolio's $\beta$ between 1.48 and 1.54.

### Fragile inputs
Drift is the benign failure; the optimizer's inputs are the serious one. Three tendencies of estimated minimum-variance and tangent portfolios recur:

* __Input sensitivity.__ The optimizer treats the estimated growth rates and covariances as exact, so estimation errors move the weights, and the tangent portfolio in particular overweights assets whose growth is overstated and underweights those whose growth is understated ([Michaud, 1989](https://doi.org/10.2469/faj.v45.n1.31), the _error maximizer_); [Chopra and Ziemba (1993)](https://doi.org/10.3905/jpm.1993.409440) showed that errors in the means matter far more than errors in the variances.
* __Concentration.__ Without binding constraints the tangent portfolio (and often the GMV portfolio, which uses only the covariance) puts most of its weight in a few names; the L6b tangent portfolio held four of thirteen firms.
* __Assumption fragility.__ The framework assumes one stable multivariate distribution, but correlations rise and volatilities jump precisely in the episodes that matter.

Two responses follow. Change the __objective__: write the investor's preferences for each asset down explicitly and let them respond to the state of the market; that is today. Update the __inputs__ as data arrive rather than freezing them on the decision date; that is L7b.
___

## Utility Maximization
Instead of asking which portfolio has the smallest variance for a target growth rate, we ask a question from consumer choice theory: given a budget and today's market, which portfolio does the investor prefer most? A __utility function__ assigns a real number to every feasible portfolio, larger for more preferred, and the investor chooses the portfolio that maximizes it subject to the budget. The proposal of this unit is to compute the preferences each trading day, from the SIM and two market inputs, so that the answer changes with the market. There are two coupled questions inside it: __which assets belong in the basket__ today, and __how much of each__; the utility functions below answer the second, and the preference model of the next section answers the first.

> __The general problem:__ At time $t$, given $|\mathcal{P}|$ assets with prices $S_{i}(t)$ (USD per share) and wealth $W_{\mathcal{P}}(t)$ (USD), choose the share counts $n_{i}\geq 0$ that maximize the investor's utility subject to the budget constraint:
> $$
\operatorname*{maximize}_{n_{1},\ldots,n_{|\mathcal{P}|}}\quad U(n_{1},\ldots,n_{|\mathcal{P}|})\qquad\text{subject to}\quad\sum_{i\in\mathcal{P}}n_{i}\,S_{i}(t) = W_{\mathcal{P}}(t)
$$
> where $U$ encodes the investor's preferences over the shares held.

This is a __holding utility__: a preference over the shares held today at today's prices, not the expected utility of an uncertain terminal wealth. Risk enters through the preferences (which will depend on the SIM betas and the market state) and through the engine's rules, not through a variance term.

### Cobb–Douglas Utility
The workhorse is the Cobb–Douglas utility, whose exponents are the preference weights.

> __Cobb–Douglas allocation:__ Let $\gamma_{i}\in(-1,1)$ be the preference weight of asset $i$. Its __sign__ classifies the assets into the preferred set $\mathcal{A}^{+} = \{i : \gamma_{i} > 0\}$, the basket, and the non-preferred set $\mathcal{A}^{-} = \{i : \gamma_{i}\leq 0\}$, held at a floor of $n_{\min}\geq 0$ shares each (a toehold of one hundredth of a share in the examples). The utility is the product of powers over the preferred set:
> $$
U(n_{1},\ldots,n_{|\mathcal{P}|}) = \prod_{i\in\mathcal{A}^{+}}n_{i}^{\gamma_{i}}\qquad(n_{i} > 0\ \text{on}\ \mathcal{A}^{+},\quad n_{i} = n_{\min}\ \text{on}\ \mathcal{A}^{-})
$$
> With $W_{\text{adj}} = W_{\mathcal{P}}(t) - n_{\min}\sum_{k\in\mathcal{A}^{-}}S_{k}(t) > 0$ the budget left after the floors, at least one preferred asset, and fractional shares, the budget-constrained maximizer is available in closed form:
> $$
\boxed{
n_{i}^{\star} = \underbrace{\frac{\gamma_{i}}{\sum_{j\in\mathcal{A}^{+}}\gamma_{j}}}_{\text{preference share}}\cdot\underbrace{\frac{W_{\text{adj}}}{S_{i}(t)}}_{\text{budget in shares}}\qquad i\in\mathcal{A}^{+},\qquad n_{i}^{\star} = n_{\min}\quad i\in\mathcal{A}^{-}\quad\blacksquare}
$$
> so the __dollar weight__ of a preferred asset within the preferred budget is its normalized preference weight, $\gamma_{i}/\sum_{j\in\mathcal{A}^{+}}\gamma_{j}$, independent of prices; as a fraction of total wealth it is that number times $W_{\text{adj}}/W_{\mathcal{P}}(t)$. The __magnitude__ of a positive $\gamma_{i}$ sets the share of the budget. If the preferred set is empty, the allocation is the floors plus cash.

Where does the closed form come from? The product of powers becomes a sum of logarithms, and the budget constraint does the rest.

> __Derivation:__ Maximizing $U$ over $\mathcal{A}^{+}$ is the same as maximizing $\ln U = \sum_{i\in\mathcal{A}^{+}}\gamma_{i}\ln n_{i}$ (the logarithm is increasing on the positive reals, so it preserves the maximizer). Introduce a Lagrange multiplier $\lambda$ for the budget over the preferred set, $\sum_{i\in\mathcal{A}^{+}}n_{i}S_{i}(t) = W_{\text{adj}}$. The first-order condition for asset $i$ is $\gamma_{i}/n_{i} = \lambda S_{i}(t)$, i.e., $n_{i}S_{i}(t) = \gamma_{i}/\lambda$: each preferred asset receives dollars in proportion to its exponent. Summing over $\mathcal{A}^{+}$ gives $W_{\text{adj}} = \sum_{j\in\mathcal{A}^{+}}\gamma_{j}/\lambda$, so $1/\lambda = W_{\text{adj}}/\sum_{j}\gamma_{j}$, and substituting back gives the boxed result. Because every $\gamma_{i} > 0$ on $\mathcal{A}^{+}$, $\ln U$ is strictly concave there and the stationary point is the unique maximum. $\blacksquare$

The closed form applies to the preferred set only: a non-positive $\gamma_{i}$ is a classification, not an exponent to be inserted (for $\gamma_{i} < 0$ the term $n_{i}^{\gamma_{i}}$ grows without bound as $n_{i}\downarrow 0$), which is why the non-preferred assets are pinned at the floor. The allocation responds immediately to a change in the preference weights: no covariance matrix, no quadratic program, no numerical solver.

### CES Utility
The Cobb–Douglas allocator always spends on every preferred asset in proportion to $\gamma_{i}$. The constant elasticity of substitution (CES) utility adds one parameter that controls how willingly the allocator trades one preferred asset for another.

> __CES allocation:__ For an elasticity of substitution $\eta > 0$, $\eta\neq 1$, the CES utility over the preferred set is:
> $$
U_{\text{CES}}(n_{1},\ldots,n_{|\mathcal{P}|}) = \left(\sum_{i\in\mathcal{A}^{+}}\gamma_{i}\,n_{i}^{(\eta-1)/\eta}\right)^{\eta/(\eta-1)}
$$
> where $\gamma_{i}$ now enters as a coefficient. Its budget-constrained maximizer is also closed form: on the preferred set the share counts are proportional to $(\gamma_{i}/S_{i}(t))^{\eta}$, the ratio $\gamma_{i}/S_{i}(t)$ being the _bang for the buck_ of consumer theory:
> $$
\boxed{
n_{i}^{\star} = W_{\text{adj}}\;\frac{\left(\gamma_{i}/S_{i}(t)\right)^{\eta}}{\sum_{j\in\mathcal{A}^{+}}S_{j}(t)\left(\gamma_{j}/S_{j}(t)\right)^{\eta}}\qquad i\in\mathcal{A}^{+}\quad\blacksquare}
$$
> Three limits frame the family: as $\eta\to 1$ the CES allocation tends to the Cobb–Douglas allocation (the utility itself is undefined at $\eta = 1$; Cobb–Douglas is its limit); as $\eta\to\infty$ the whole preferred budget goes to the asset with the largest $\gamma_{i}/S_{i}(t)$ (when that asset is unique); as $\eta\to 0^{+}$ the allocator buys equal share counts of every preferred asset regardless of $\gamma_{i}$ or price. The elasticity is a __concentration dial__. The proofs are in the optional advanced notebook, together with a caveat: because CES works with share counts, for $\eta\neq 1$ its dollar weights depend on the price level of each share, which Cobb–Douglas weights do not.

The engine's CES variant ties the elasticity to the market-state signal $\xi_{t}$ (an EMA crossover on SPY, positive when the market has been falling, defined in the next section), so that the allocator concentrates when the signal is quiet and diversifies when it is loud in either direction:
$$
\eta(\xi_{t}) = \eta_{\min} + \frac{\eta_{\max} - \eta_{\min}}{1 + |\xi_{t}|}\qquad 0 < \eta_{\min}\leq\eta_{\max}
$$
with $\eta_{\min} = 0.5$ and $\eta_{\max} = 5$ in the examples: at $\xi_{t} = 0$ the elasticity is $\eta_{\max}$, and as $|\xi_{t}|$ grows it falls toward $\eta_{\min}$. The log-linear utility $\sum_{i\in\mathcal{A}^{+}}\gamma_{i}\ln n_{i}$ is the logarithm of Cobb–Douglas, same allocation with additive values (advanced notebook).

> __Example__
>
> [▶ Building a maximum-utility portfolio allocator](CHEME-5660-L7a-Example-Utility-Allocator-Fall-2026.ipynb). On January 6, 2025, the first day of the year on which the basket was nonempty, four of the thirteen firms are preferred (AAPL, MSFT, AMD, NVDA, the tangent portfolio's own support), the Cobb–Douglas weights are NVDA 0.59, AMD 0.15, AAPL 0.13, MSFT 0.12 with the floors taking eleven dollars of the thousand, and raising the CES elasticity to $\eta = 5$ puts 99% of the budget in NVDA.
___

## Preference Weights from the Single Index Model
The allocators need the preference weights $\gamma_{i}(t)$, and it is here that the two coupled questions are answered. The weights come from the SIM parameters $(\alpha_{i}, \beta_{i})$ of L6a and from two numbers that describe the market on day $t$: the __recent market growth rate__ $\tilde{g}_{M,t}$, which plays the role of the SIM's expected market growth, and a __market-state signal__ $\xi_{t}$ that says whether the market has recently been rising or falling.

> __Preference model:__ For each asset $i$ on day $t$:
> $$
\boxed{
\gamma_{i}(t) = \tanh\left(\frac{\alpha_{i}}{\beta_{i}^{\xi_{t}}} + \beta_{i}^{1-\xi_{t}}\,\tilde{g}_{M,t}\right)\quad\blacksquare}
$$
> The hyperbolic tangent bounds the weight in $(-1, 1)$. Its argument is the SIM's expected growth rate of asset $i$ in today's market, $\alpha_{i} + \beta_{i}\tilde{g}_{M,t}$ (units: inverse years, read as a pure number at the one-year horizon), seen through a __regime lens__: the identity $\alpha_{i}/\beta_{i}^{\xi_{t}} + \beta_{i}^{1-\xi_{t}}\tilde{g}_{M,t} = (\alpha_{i} + \beta_{i}\tilde{g}_{M,t})/\beta_{i}^{\xi_{t}}$ shows that the signal divides that expectation by $\beta_{i}^{\xi_{t}}$.

The model answers the two questions separately:

* __Which assets belong in the basket?__ Because $\beta_{i}^{\xi_{t}} > 0$ and $\tanh$ preserves sign, the __sign__ of $\gamma_{i}(t)$ is the sign of $\alpha_{i} + \beta_{i}\tilde{g}_{M,t}$: asset $i$ is preferred exactly when its expected growth in today's market is positive, i.e., when $\tilde{g}_{M,t} > -\alpha_{i}/\beta_{i}$. As the recent market growth moves, assets cross between the sets: a strong market pulls in firms with a small negative $\alpha_{i}$ and a large $\beta_{i}$, a weak market pushes out everything but the strongest intercepts, and when every $\gamma_{i}(t)\leq 0$ the model is saying that __this basket does not belong in this market__: the allocator holds the floors plus cash, and an investor who is not content with cash should rethink which assets are in the basket (the ticker-picking methods of L12b and L13a are that rethink; the L7b update of $\alpha_{i}, \beta_{i}$ moves the thresholds too).
* __How much of each?__ The __magnitude__ of a positive $\gamma_{i}(t)$ sets its share of the budget, and the lens moves the magnitudes: a bearish signal ($\xi_{t} > 0$) divides by $\beta_{i}^{\xi_{t}} > 1$ for high-beta names and by a number below one for low-beta names, tilting the budget toward low-beta names, all else equal; a bullish signal ($\xi_{t} < 0$) tilts it toward high-beta names.

Three assumptions sit behind the box, and we state them rather than leave them implicit.

> __Assumptions, stated:__ (i) $\beta_{i} > 0$ for every asset in the universe, because $\beta_{i}$ is raised to a real power (the thirteen course firms have $\hat{\beta}_{i}$ between 0.54 and 1.75, and the examples assert it; a universe with a non-positive beta needs a rule for that asset, for example assignment to $\mathcal{A}^{-}$). (ii) The argument is a growth rate in inverse years read as a pure number at the one-year horizon, which is why $\alpha_{i}$ and $\tilde{g}_{M,t}$ enter in annualized units. (iii) The recent market growth $\tilde{g}_{M,t}$ is a short exponential moving average of the daily market growth rates and therefore swings by about a unit per year, an order of magnitude more than the intercepts: that is what makes the basket respond to the market, and it is also why the engine's schedule and rules, not the model, decide how often the response is acted on.

The two market inputs are computed from the market index (SPY) with exponential moving averages. For a price series $S_{t}$ and a window of $L$ trading days, the EMA is the recursion $\bar{S}_{t} = \omega\,S_{t} + (1-\omega)\,\bar{S}_{t-1}$ with weight $\omega = 2/(L+1)$.

> __The market inputs:__ With a short window $L_{\text{short}} = 21$ that tracks recent momentum, a long window $L_{\text{long}} = 63$ that tracks the trend, and a dimensionless gain $G > 0$ that sets the scale, the market-state signal is the scaled crossover of the two price EMAs:
> $$
\xi_{t} = -G\left(\frac{\bar{S}^{\text{short}}_{t}}{\bar{S}^{\text{long}}_{t}} - 1\right)
$$
> A __falling__ market (short EMA below the long EMA) gives $\xi_{t} > 0$, bearish; a __rising__ market gives $\xi_{t} < 0$, bullish. We choose $G$ on the training data so that $|\xi_{t}|\leq 1$ on 99% of training days (the reciprocal of the 99th percentile of the absolute crossover after a warm-up of $t_{0} = L_{\text{short}} + L_{\text{long}}$ days); on the course data $G\approx 21$. The recent market growth is the EMA, window $L_{\text{growth}} = 10$, of the daily market growth rates $g_{M,t} = \ln(S_{t}/S_{t-1})/\Delta{t}$ (inverse years).

Both inputs on day $t$ must be computed from prices __through day $t-1$__; the engine below explains why. On the last training day the signal was $-0.28$ (mildly bullish) and the recent market growth $-0.76$ per year (SPY fell through late December 2024), below every firm's threshold, so on the first trading day of 2025 no firm was preferred. Signals with more information, for example a per-firm news term added to the argument, are how the preference model grows in L15b.
___

## The Rebalancing Engine
The allocator answers "what would I hold right now?" The engine asks it on a schedule and decides whether, and how much, to trade. Each bar (trading day) is one pass through four stages: __observe__ the new bar (execution prices, the lagged market inputs, the SIM parameters), __orient__ (compute the preference weights), __decide__ (compute the targets and apply the rules), and __act__ (trade at today's price, pay the cost, update the account, record). A static allocation is one pass on the decision date followed by nothing.

<div>
    <center>
        <img src="figs/Fig-L7a-Rebalancing-Engine-Loop.svg" width="900" alt="The rebalancing engine as one pass per trading day: observe (execution prices, lagged market inputs, SIM parameters), orient (preference weights), decide (Cobb-Douglas targets and the schedule, turnover cap, and drawdown rules), act (trade at today's close, pay the cost, update cash and wealth), then the next bar."/>
    </center>
</div>

### The account
The engine keeps a self-financing account: cash plus positions, with every dollar of cost taken out of the book. Let $n_{i,t}$ be the shares of asset $i$ held after the close of day $t$ and $\text{cash}_{t}$ the cash after that close. Cash earns the risk-free growth rate, so during day $t$ it grows by $e^{g_{f}\Delta{t}}$, and at the close the book, before any trade, is worth:
$$
W^{-}_{\mathcal{P}}(t) = \underbrace{\text{cash}_{t-1}\,e^{g_{f}\Delta{t}}}_{\text{cash}} + \underbrace{\sum_{i\in\mathcal{P}}n_{i,t-1}\,S_{i}(t)}_{\text{positions marked to market}}
$$
where the superscript minus means "before today's trades". If the engine trades share deltas $\Delta{n}_{i} = n_{i,t} - n_{i,t-1}$ at the close prices $S_{i}(t)$, the gross notional traded is $Q_{t} = \sum_{i}|\Delta{n}_{i}|\,S_{i}(t)$, and with a cost rate $c$ (dollars of cost per dollar traded; five basis points, $c = 5\times 10^{-4}$, in the examples) the transaction cost is $C_{t} = c\,Q_{t}$. The cost comes out of the book, and the new positions are sized against what is left:
$$
\boxed{W^{+}_{\mathcal{P}}(t) = W^{-}_{\mathcal{P}}(t) - C_{t}}
$$
Because the cost depends on the trade and the trade on the post-cost budget, the engine solves the two together (a fixed-point iteration that converges in a few steps), so that $\text{cash}_{t} + \sum_{i}n_{i,t}S_{i}(t) = W^{+}_{\mathcal{P}}(t)$ holds exactly and cash never goes negative. Sizing against $W^{-}_{\mathcal{P}}(t)$ and debiting the cost afterward is the tempting shortcut; it leaves the account short by $C_{t}$.

### Timing
The market inputs and the preference weights that drive the decision on day $t$ are computed from prices __through day $t-1$__, and the trade executes at the close of day $t$ (a market-on-close order fills at the closing price, which is known when it fills). This is __next-bar execution__: nothing in the decision uses information that was not available before the bar it trades on. Computing the inputs from today's close and trading at today's close would let the engine act on a price move it could not have known about, a look-ahead that inflates every backtest it touches; the second example asserts the lag in code.

### Three trigger rules
Left alone, the loop would trade every day (expensive), ignore a crash (catastrophic), or act on every wiggle of the market inputs. Three rules bound it:

| Rule | Parameter | What it does |
|:--|:--|:--|
| Reallocation schedule | the set of trading days $\mathcal{R}$ | The engine may trade only on days $t\in\mathcal{R}$: daily, or every 21 trading days (about monthly). On other days it holds. |
| Turnover cap | $\tau_{\max}$ | With $\Delta{w}_{i}$ the change of asset $i$'s weight in the rebalance (as a fraction of $W^{-}_{\mathcal{P}}(t)$) and $\Delta{w}_{\text{cash}} = -\sum_{i}\Delta{w}_{i}$ the cash leg, the __one-way turnover__ is $\tau_{t} = \tfrac{1}{2}\left(\sum_{i}\lvert\Delta{w}_{i}\rvert + \lvert\Delta{w}_{\text{cash}}\rvert\right)$, the fraction of wealth that changes hands one way. If a proposed move has $\tau_{t} > \tau_{\max}$, it is scaled down to the cap. The cost applies to the __gross__ notional $Q_{t}$, which is twice the one-way turnover times wealth when cash does not move. |
| Drawdown limit | $d_{\max}$ | A circuit breaker checked every day, before the schedule: if wealth is more than $d_{\max}$ below its running peak, $1 - W^{-}_{\mathcal{P}}(t)/\max_{s\leq t}W^{-}_{\mathcal{P}}(s) > d_{\max}$, the book is liquidated to cash (bypassing the cap), the engine stays in cash for $n_{\text{lock}}$ more days, and on its first scheduled trade after that the peak is rebased to the current wealth. Because it is checked once a day after the loss and the liquidation pays a cost, it does not cap the recorded drawdown at $d_{\max}$. |

The rules interact. The cap applies to every scheduled trade, including the first: moving all of the wealth from cash into risky assets is a one-way turnover of one, so with $\tau_{\max} = 0.5$ the engine is only half invested until its next scheduled day, at the start and after every return from the floors or from a lock. The second example shows it.

### Algorithm: Rebalancing Engine (Backtest)
The engine is allocator-agnostic: it takes the price matrix and a __target function__ that returns the desired risky weights on a scheduled day, and our target functions wrap the allocators of this lecture around the lagged preference weights.

__Initialize:__ Given the close prices $S_{i}(t)$ for $t = 1,\ldots,N$ trading days and $i\in\mathcal{P}$, the initial wealth $W_{\mathcal{P}}(0)$ held in cash, the reallocation set $\mathcal{R}$, the rule parameters $(c, \tau_{\max}, d_{\max}, n_{\text{lock}})$, and a target function $\mathbf{w}^{\star}(t, \text{state})$ that returns risky weights ($w_{i}^{\star}\geq 0$, $\sum_{i}w_{i}^{\star}\leq 1$, the remainder cash) given the day and the current state (pre-trade wealth, weights, shares, cash, and a copy of the day's execution prices). Our target functions compute $\gamma_{i}(t)$ from the market inputs through day $t-1$ and return the Cobb–Douglas (or CES) allocation at the day's closing prices as weights; the Cobb–Douglas weights do not depend on those prices, the CES weights do (through the bang for the buck), which is a same-close price dependence, not a signal dependence.

For $t = 1,\ldots,N$ __do__:
1. Accrue the cash by $e^{g_{f}\Delta{t}}$, mark the positions to market at $S_{i}(t)$ to get $W^{-}_{\mathcal{P}}(t)$, and update the running peak.
2. Circuit breaker: if the drawdown exceeds $d_{\max}$ and the book holds risky positions, liquidate at $S_{i}(t)$, pay $c$ on the notional sold, lock for $n_{\text{lock}}$ more days, and go to step 5.
3. If $t\in\mathcal{R}$ and the engine is not locked: ask the target function for $\mathbf{w}^{\star}$ (rebasing the peak if this is the first trade after a lock); form the proposed change of weights and scale it to the cap if $\tau_{t} > \tau_{\max}$.
4. Solve the cost and the positions together on the post-cost budget, execute $\Delta{n}_{i}$ at $S_{i}(t)$, pay $C_{t} = c\,Q_{t}$, and update the cash.
5. Record wealth, cash, positions, weights, turnover, cost, whether the engine traded or was locked, and the count of interventions.

__Output:__ The histories, indexed by trading day, that every scorecard consumes. This is a __backtest__: it replays a complete price history one bar at a time under the timing rule; the same loop runs live by replacing the replay with a data feed. The static baselines we compare against, the buy-and-hold GMV, tangent, and equal-weight portfolios and SPY, are the same engine with a constant target and $\mathcal{R} = \{1\}$, so they pay the same costs and follow the same accounting.

> __Example__
>
> [▶ Running the adaptive rebalancing engine through 2025](CHEME-5660-L7a-Example-Adaptive-Rebalancing-Scorecard-Fall-2026.ipynb). Over 2025 the model classified the whole basket as non-preferred on 57 of 250 days and preferred all thirteen firms on 152. With $\mathcal{R}$ every 21 days, $\tau_{\max} = 0.5$, $d_{\max} = 0.10$, and $c = 5$ basis points, the Cobb–Douglas engine made 12 scheduled trades, held only the floors plus cash (risky exposure below 2% of wealth) on 50 days, never fired the breaker, and ended the year at 1.53 times its initial wealth with a maximum drawdown of 9.0%; the daily schedule made 228 trades, fired the breaker once, paid twenty times the cost, and ended at 1.40; the buy-and-hold tangent portfolio ended at 1.28 with a 31% drawdown, GMV at 1.38 (13%), equal weights at 1.59 (24%), and SPY at 1.17 (19%).
___

## Scoring a Realized Path
How do we say whether one run of the engine did well? The first question is economic: did the portfolio beat what the same dollars would have earned at the risk-free growth rate? For one initial outflow, one terminal value, and a constant $g_{f}$ over the horizon $T = N\Delta{t}$ years, that is the __portfolio NPV__:
$$
\boxed{\text{NPV}(g_{f}, T) = \underbrace{-W_{\mathcal{P}}(0)}_{\text{investment}} + \underbrace{W_{\mathcal{P}}(T)\,e^{-g_{f}T}}_{\text{discounted terminal wealth}}}
$$
A positive NPV means the run beat holding cash at $g_{f}$; the risk-free portfolio itself has NPV zero, and it is the zero every risky run is measured against. The scorecard is one row per run and these columns:

| Metric | Definition | Reads as |
|:--|:--|:--|
| Terminal wealth, NPV | $W_{\mathcal{P}}(T)$; $-W_{\mathcal{P}}(0) + W_{\mathcal{P}}(T)e^{-g_{f}T}$ | did the run beat cash at $g_{f}$ |
| Realized growth | $\ln\left(W_{\mathcal{P}}(T)/W_{\mathcal{P}}(0)\right)/T$ (inverse years) | the growth rate the year delivered |
| Maximum drawdown | $\max_{t}\left(1 - W_{\mathcal{P}}(t)/\max_{s\leq t}W_{\mathcal{P}}(s)\right)$ | the worst peak-to-trough decline along the path |
| Realized Sharpe ratio | mean of the daily excess log growth $\ln(W_{\mathcal{P}}(t)/W_{\mathcal{P}}(t-1)) - g_{f}\Delta{t}$ over its standard deviation, times $\sqrt{1/\Delta{t}}$ (annualized, as in L6b) | growth per unit of realized risk |
| Turnover, cost, interventions | total one-way turnover $\sum_{t}\tau_{t}$; total cost $\sum_{t}C_{t}$; breaker firings | what it cost to move, and how often the breaker acted |

The static portfolios pay one round of cost and have no interventions; the engine pays to move. Everything on the scorecard describes __one realized path__: 2025 happened once. It says what each policy did, not how likely that outcome was; a fail rate (how often the portfolio ends below the risk-free baseline), a lower quantile of terminal wealth, or the average of the worst outcomes need many paths, which is L7b.
___

## Optional Advanced Material
The notebook below extends today's material. It is optional and is not a prerequisite for L7b; the [advanced index](advanced/README.md) describes it.

* [▶ CES limits, the elasticity rule, and log-linear utility](advanced/adaptive_utility/CHEME-5660-L7a-Advanced-CES-Limits-Fall-2026.ipynb). Derive the CES closed form and prove its three limits, read the two limits of the elasticity rule $\eta(\xi_{t})$, show that log-linear utility has the Cobb–Douglas optimizer, and see why the CES allocation over share counts depends on the price level of each share when $\eta\neq 1$.
___

## Summary
In this lecture, we treated the portfolio as a process: we derived how a static allocation drifts and why its inputs are fragile, replaced the minimum-variance objective with explicit preferences and the Cobb–Douglas and CES allocators, computed the preferences from the single index model and two market inputs so that they choose the basket and the weights within it, and wrapped the allocator in a rebalancing engine with a self-financing account, next-bar execution, three trigger rules, and a scorecard.

> __Key Takeaways:__
>
> * __Utility makes preferences explicit and Cobb–Douglas turns them into weights in closed form:__ the sign of a preference weight puts an asset in or out of the basket, its magnitude sets the asset's share of the preferred budget as the normalized preference weight, independent of prices, and the CES elasticity is a concentration dial with Cobb–Douglas as its unit-elasticity limit.
> * __The SIM plus two market inputs supplies the preferences, and the rules bound the response:__ the recent market growth decides, asset by asset, whether the SIM's expected growth in today's market is positive and so which assets belong in the basket (an empty basket is the model saying to rethink the basket or hold cash), the crossover signal tilts the magnitudes between high-beta and low-beta names, and the schedule, turnover cap, and drawdown limit decide when, how far, and whether the engine acts, with the cost taken out of the book as the trade is sized.
> * __A realized-path scorecard is one draw:__ portfolio NPV against cash, drawdown, realized growth and Sharpe ratio, turnover, cost, and interventions say what a policy did on the year that happened; the distribution of what it could have done needs many simulated paths.

Next time, in L7b, we let the SIM parameters themselves update as data arrive, replay the engine with frozen and with updated parameters on the realized year and on an ensemble of simulated futures, and read the distributional scorecard those paths allow.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___